# RappiPlus Strategic Business Analysis

This notebook documents an end-to-end business analysis of RappiPlus. It combines data quality validation, profitability analysis, SQL-based customer behavior analysis, cohort retention, and A/B testing to support commercial and product decisions.

The project connects transactional, catalog, marketing, behavioral, and experimental data. Its purpose is not only to calculate metrics, but to translate them into decisions about profitability, customer retention, conversion, product performance, and marketing investment.


## 1. Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**Objective:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd

In [ ]:
# cargar archivos
orders = pd.read_csv('rappiplus_orders_raw.csv')
catalog = pd.read_csv('rappiplus_catalog.csv')
marketing = pd.read_csv('rappiplus_marketing_spend.csv')

In [ ]:

# Explorar datasets

datasets = {
    'orders': orders,
    'catalog': catalog,
    'marketing': marketing
}

for nombre, df in datasets.items():
    print(f'\n===== {nombre.upper()} =====')
    print('Dimensiones:', df.shape)
    print('Columnas:', df.columns.tolist())
    display(df.head())
    df.info()


### Revisión y calidad de datos

**Objective:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [ ]:
# Diagnóstico inicial de calidad
for nombre, df in {
    'orders': orders,
    'catalog': catalog,
    'marketing': marketing
}.items():
    print(f'\n===== {nombre.upper()} =====')
    print('Valores faltantes:')
    print(df.isna().sum())
    print('\nFilas duplicadas:', df.duplicated().sum())
    print('\nValores mínimos numéricos:')
    print(df.select_dtypes(include='number').min())

In [ ]:

# Validar y convertir fechas al formato correcto
orders = pd.read_csv('rappiplus_orders_raw.csv')
marketing = pd.read_csv('rappiplus_marketing_spend.csv')

orders.columns = orders.columns.str.strip()
marketing.columns = marketing.columns.str.strip()

fecha_orders = next(col for col in orders.columns if 'fecha' in col.lower())
fecha_marketing = next(col for col in marketing.columns if 'fecha' in col.lower())

orders[fecha_orders] = pd.to_datetime(orders[fecha_orders], errors='coerce')
marketing[fecha_marketing] = pd.to_datetime(marketing[fecha_marketing], errors='coerce')

print('Columna de fecha en orders:', fecha_orders)
print('Tipo:', orders[fecha_orders].dtype)
print('Fechas inválidas:', orders[fecha_orders].isna().sum())

print('\nColumna de fecha en marketing:', fecha_marketing)
print('Tipo:', marketing[fecha_marketing].dtype)
print('Fechas inválidas:', marketing[fecha_marketing].isna().sum())

In [ ]:
orders[(orders['cantidad'] <= 0) | (orders['monto_total'] < 0)]

In [ ]:
orders = orders[
    (orders['cantidad'] > 0) &
    (orders['monto_total'] >= 0)
].copy()

print('Registros inválidos restantes:')
print(orders[(orders['cantidad'] <= 0) | (orders['monto_total'] < 0)].shape[0])
print('Dimensiones después de limpiar:', orders.shape)

In [ ]:
marketing['canal'] = marketing['canal'].fillna('Desconocido')

print('Valores nulos restantes en canal:')
print(marketing['canal'].isna().sum())

In [ ]:
monto_esperado = (
    orders['cantidad'] * orders['precio_unitario']
    - orders['monto_descuento']
)

diferencia_monto = (
    orders['monto_total'] - monto_esperado
).abs().round(2)

montos_incoherentes = orders[diferencia_monto > 0.01]

print('Registros con montos incoherentes:')
print(montos_incoherentes.shape[0])
print('Diferencia máxima encontrada:')
print(diferencia_monto.max())

---
**Data export:** Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

## 2. Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**Objective:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [ ]:

# Completar limpieza de datos
orders = orders.drop_duplicates().copy()

# Eliminar pedidos cuyo producto y costo no pueden identificarse
orders = orders.dropna(subset=['nombre_producto']).copy()

# Corregir valores categóricos
orders['pais'] = (
    orders['pais']
    .str.strip()
    .str.title()
    .fillna('Desconocido')
)

orders['dispositivo'] = orders['dispositivo'].fillna('Desconocido')
orders['fuente_referencia'] = orders['fuente_referencia'].fillna('Desconocido')

orders['categoria_producto'] = orders['categoria_producto'].replace(
    {'Electronica': 'Electrónica'}
)

print('Dimensiones finales:', orders.shape)
print('Duplicados restantes:', orders.duplicated().sum())
print('Productos faltantes:', orders['nombre_producto'].isna().sum())
print('Categorías:', orders['categoria_producto'].unique())


In [ ]:
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

print('Archivos limpios actualizados correctamente.')

In [ ]:
# Calcular KPIs principales
orders_kpi = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)

ingreso_total = orders_kpi['monto_total'].sum()
costo_total = (
    orders_kpi['cantidad'] * orders_kpi['costo_unitario']
).sum()
ganancia_total = ingreso_total - costo_total
margen_ganancia = (ganancia_total / ingreso_total) * 100

print(f'Ingreso total: ${ingreso_total:,.2f}')
print(f'Costo total: ${costo_total:,.2f}')
print(f'Ganancia total: ${ganancia_total:,.2f}')
print(f'Margen de ganancia: {margen_ganancia:.2f}%')

In [ ]:
# Incluir marketing y calcular profit final
gasto_marketing = marketing['gasto'].sum()
profit_total = ingreso_total - costo_total - gasto_marketing
margen_neto = (profit_total / ingreso_total) * 100

print(f'Gasto total en marketing: ${gasto_marketing:,.2f}')
print(f'Profit total: ${profit_total:,.2f}')
print(f'Margen neto: {margen_neto:.2f}%')
print('¿El negocio es rentable?:', 'Sí' if profit_total > 0 else 'No')

In [ ]:
# Parte 2: Comportamiento de ventas

ventas_por_pedido = orders.groupby('id_pedido').agg(
    ticket=('monto_total', 'sum'),
    productos=('cantidad', 'sum')
)

ticket_promedio = ventas_por_pedido['ticket'].mean()
productos_promedio = ventas_por_pedido['productos'].mean()

ventas_producto = (
    orders.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)

producto_mas_vendido = ventas_producto.index[0]
unidades_producto_top = ventas_producto.iloc[0]

marketing_por_canal = (
    marketing.groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)

print(f'Ticket promedio por orden: ${ticket_promedio:,.2f}')
print(f'Cantidad promedio de productos por orden: {productos_promedio:.2f}')
print(f'Producto más vendido: {producto_mas_vendido}')
print(f'Unidades vendidas del producto líder: {unidades_producto_top:,.0f}')

print('\nGasto de marketing por canal:')
print(marketing_por_canal)

## 3. Entender dónde se pierden los usuarios (funnel de conversión)

**Objective:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
import os

# Database credentials are read from environment variables and are never stored
# in the public repository.
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT', '5432'),
    'db': os.getenv('DB_NAME')
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

In [ ]:


# PARTE 1: Totales del funnel

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN 5
        WHEN 'purchase' THEN 6
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

In [ ]:
# PARTE 2: Conversiones

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'select_item' THEN 2
            WHEN 'add_to_cart' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS orden
    FROM events
    GROUP BY nombre_evento
),
conversiones AS (
    SELECT
        nombre_evento,
        usuarios_unicos,
        orden,
        LAG(usuarios_unicos) OVER (ORDER BY orden) AS usuarios_etapa_anterior
    FROM funnel
)
SELECT
    nombre_evento,
    usuarios_unicos,
    ROUND(
        usuarios_unicos * 100.0 / usuarios_etapa_anterior,
        2
    ) AS conversion_etapa_pct
FROM conversiones
ORDER BY orden;
'''

conversion = pd.read_sql(query_conversion, con=engine)

conversion_total = (
    totals.loc[totals['nombre_evento'] == 'purchase', 'usuarios_unicos'].iloc[0]
    / totals.loc[totals['nombre_evento'] == 'first_visit', 'usuarios_unicos'].iloc[0]
    * 100
)

display(conversion)
print(f'Conversión total del funnel: {conversion_total:.2f}%')

## 4. Evaluar si los usuarios regresan (retención por cohortes)

**Objective:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Retención por cohortes

query_cohort_retention_final = '''
WITH usuarios_cohorte AS (
    SELECT
        id_usuario,
        TO_CHAR(CAST(fecha_registro AS DATE), 'YYYY-MM') AS cohorte
    FROM users
),
retencion AS (
    SELECT
        u.cohorte,
        COUNT(DISTINCT u.id_usuario) AS clientes_iniciales,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 7
                 AND a.activo = 1
            THEN a.id_usuario
        END) AS retenido_w1,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 14
                 AND a.activo = 1
            THEN a.id_usuario
        END) AS retenido_w2,

        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 21
                 AND a.activo = 1
            THEN a.id_usuario
        END) AS retenido_w3

    FROM usuarios_cohorte AS u
    LEFT JOIN user_activity AS a
        ON u.id_usuario = a.id_usuario
    GROUP BY u.cohorte
)

SELECT
    cohorte,
    clientes_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(retenido_w1 * 100.0 / clientes_iniciales, 2) AS semana_1,
    ROUND(retenido_w2 * 100.0 / clientes_iniciales, 2) AS semana_2,
    ROUND(retenido_w3 * 100.0 / clientes_iniciales, 2) AS semana_3
FROM retencion
ORDER BY cohorte;
'''

cohorte_final = pd.read_sql(
    query_cohort_retention_final,
    con=engine
)

cohorte_final

In [ ]:

query_user_activity = '''
SELECT *
FROM user_activity;
'''

user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

## 5. Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

### Hipótesis estadística

- **H₀ (Hipótesis nula):** La tasa de conversión del grupo control es igual a la tasa de conversión del grupo tratamiento.
- **H₁ (Hipótesis alternativa):** La tasa de conversión del grupo control es diferente a la tasa de conversión del grupo tratamiento.

**Test estadístico:** Prueba Z de dos proporciones  
**Nivel de significancia alpha:** 0.05

### Resultado y recomendación

Como el valor p es 0.4161 y es mayor que 0.05, no se rechaza H₀. No existe evidencia estadísticamente significativa de una diferencia entre las tasas de conversión del grupo control y tratamiento.

Aunque el tratamiento obtuvo una conversión ligeramente superior (16.29 % frente a 15.69 %), no se recomienda implementarlo de manera general con base en este experimento. Se recomienda mantener la versión actual y realizar una nueva prueba con otras mejoras o un periodo de evaluación mayor.

In [ ]:
# Explorar dataset del experimento

experiment = pd.read_csv('experiment_checkout_ui.csv')

print('Dimensiones:', experiment.shape)
display(experiment.head())
print('\nInformación de columnas:')
experiment.info()



In [ ]:
# Comparar tasas de conversión por variante

resumen_conversion = (
    experiment.groupby('variante')['convirtio']
    .agg(['count', 'sum', 'mean'])
)

resumen_conversion['tasa_conversion_pct'] = (
    resumen_conversion['mean'] * 100
).round(2)

resumen_conversion

In [ ]:
# Prueba Z de dos proporciones

from statsmodels.stats.proportion import proportions_ztest

exitos = resumen_conversion.loc[
    ['control', 'tratamiento'], 'sum'
].to_numpy()

observaciones = resumen_conversion.loc[
    ['control', 'tratamiento'], 'count'
].to_numpy()

z_stat, p_valor = proportions_ztest(
    count=exitos,
    nobs=observaciones,
    alternative='two-sided'
)

alpha = 0.05

print(f'Estadístico Z: {z_stat:.4f}')
print(f'Valor p: {p_valor:.4f}')

if p_valor < alpha:
    print('Conclusión: se rechaza H₀. Existe una diferencia significativa.')
else:
    print('Conclusión: no se rechaza H₀. No existe una diferencia significativa.')

## 6. Comunicar los resultados (Dashboard en BI)

**Objective:**  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc